# Module 10: Unique Distributed ID Generation Snowflake — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/snowflake_generator.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import snowflake_generator

classes = [n for n, o in inspect.getmembers(snowflake_generator, inspect.isclass)
           if o.__module__ == 'snowflake_generator']
functions = [n for n, o in inspect.getmembers(snowflake_generator, inspect.isfunction)
             if o.__module__ == 'snowflake_generator']

print('module   : snowflake_generator')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(snowflake_generator, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Snowflake id monotonicity

This is the module's own `test_snowflake_id_monotonicity` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import concurrent.futures

from snowflake_generator import (
    SnowflakeGenerator,
)

gen = SnowflakeGenerator(datacenter_id=1, worker_id=1)
ids = [gen.next_id() for _ in range(100)]

# Strictly increasing order
for i in range(len(ids) - 1):
    assert ids[i] < ids[i + 1]

print('PASSED: test_snowflake_id_monotonicity')

## 3. 🔮 Prediction — commit before you run

Two Snowflake generators on the same machine, same millisecond. Predict whether they can emit the same ID, and which field prevents it.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_snowflake_bit_packing_and_parsing`, which tests exactly this property.


In [ ]:
gen = SnowflakeGenerator(datacenter_id=7, worker_id=15)
raw_id = gen.next_id()

parsed = gen.parse_id(raw_id)
assert parsed.id == raw_id
assert parsed.datacenter_id == 7
assert parsed.worker_id == 15
assert parsed.sequence >= 0
assert parsed.timestamp_ms > gen.epoch

print('PASSED: test_snowflake_bit_packing_and_parsing')

## 4. Measure it: Concurrent id uniqueness across threads

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_concurrent_id_uniqueness_across_threads` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

gen = SnowflakeGenerator(datacenter_id=2, worker_id=4)
total_ids = 5000

def generate_batch(count: int) -> list[int]:
    return [gen.next_id() for _ in range(count)]

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(generate_batch, 625) for _ in range(8)]
    all_ids = []
    for f in concurrent.futures.as_completed(futures):
        all_ids.extend(f.result())

# Every single ID generated must be strictly unique!
assert len(all_ids) == total_ids
assert len(set(all_ids)) == total_ids

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_concurrent_id_uniqueness_across_threads')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(snowflake_generator) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Coordination-free unique IDs need time, machine identity, and a sequence field.
2. Clock skew is the failure mode; a monotonic guard is not optional.
3. Sortable IDs give you index locality for free - a real database benefit.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
